### NAME : RUTUJA KALE
### U23CS095
### ASSIGNMENT 6

In [1]:
!pip install gensim nltk scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 56.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

import gensim.downloader as api

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
!pip install kaggle

In [5]:
!pip install -q kaggle

In [6]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"rutuja0904","key":"018a0be4c5240af728215240d6a6ef79"}'}

In [7]:
!ls

'kaggle (1).json'   sample_data


In [8]:
!mkdir -p ~/.kaggle
!mv "kaggle (1).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle datasets download -d kritanjalijain/amazon-reviews

Dataset URL: https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews
License(s): CC0-1.0
 97% 1.25G/1.29G [00:16<00:01, 47.0MB/s]
100% 1.29G/1.29G [00:16<00:00, 85.1MB/s]


In [10]:
!unzip -q amazon-reviews.zip

In [11]:
!ls

amazon_review_polarity_csv.tgz	sample_data  train.csv
amazon-reviews.zip		test.csv


In [12]:
# load dataset
train = pd.read_csv("train.csv", header=None)
test = pd.read_csv("test.csv", header=None)

train.columns = ["label","title","review"]
test.columns = ["label","title","review"]

In [13]:
# craete full text
train["text"] = train["title"] + " " + train["review"]

In [14]:
# sampling
train = train.sample(50000, random_state=42)

In [15]:
# text preprocessing
# create stopwords
stop_words = set(stopwords.words("english"))

In [16]:
# function
def preprocess(text):

    text = str(text)
    text = text.lower()

    text = re.sub(r"[^a-z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return tokens

In [18]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
# applying
processed_reviews = train["text"].apply(preprocess)

In [20]:
# load word2vec
word2vec = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [21]:
# load fastext : represents character n-grams, allowing it to handle unseen or misspelled words.
fasttext = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [24]:
# create review embeddings
# fastext embedding function
def fasttext_embedding(tokens):

    vectors = []

    for word in tokens:
        if word in fasttext.key_to_index:   # check vocabulary
            vectors.append(fasttext[word])

    if len(vectors) == 0:
        return np.zeros(300)

    return np.mean(vectors, axis=0)

In [25]:
# generate embeddings
review_embeddings = np.vstack(
    processed_reviews.apply(fasttext_embedding)
)

In [26]:
# embed user query
query = "this is responsible ai"
query_tokens = preprocess(query)

query_vector = fasttext_embedding(query_tokens)

In [27]:
# cosine similarity : measures angle between vectors
similarities = cosine_similarity(
    [query_vector],
    review_embeddings
)[0]

In [28]:
# retrieve top 5 reviews
top5_idx = similarities.argsort()[-5:][::-1]

top_reviews = train.iloc[top5_idx]["text"]
print(top_reviews)

656006     TRACK LISTING TRACKS:1. Di Bolina2. Nel Bene E...
1221524    L'avventura dei sentimenti "L'avventura" inizz...
3254179    SUPER COLECTIE! CELE DOUA FIGURINE ARATA FOART...
1965146    Excelente!!! Adorei o produto!!! Ele é lindo e...
1607553    Magnifico, elegante, solido. Questo orologio h...
Name: text, dtype: object


In [31]:
# introduce noise : show FastText handles typos better
def add_noise(text):

    text = str(text)   # convert to string

    words = text.split()

    if len(words) > 2:
        words[1] = words[1][:-1] + "x"

    return " ".join(words)

In [32]:
# applying
noisy_reviews = train["text"].apply(add_noise)

In [33]:
# preprocess
noisy_processed = noisy_reviews.apply(preprocess)

In [34]:
# create embeddings
# word2vec
def word2vec_embedding(tokens):

    vectors = []

    for word in tokens:
        if word in word2vec:
            vectors.append(word2vec[word])

    if len(vectors) == 0:
        return np.zeros(300)

    return np.mean(vectors, axis=0)

In [35]:
# fastext
fasttext_vectors = noisy_processed.apply(fasttext_embedding)
word2vec_vectors = noisy_processed.apply(word2vec_embedding)

In [36]:
# compare similarity results again
fasttext_sim = cosine_similarity(
    [query_vector],
    np.vstack(fasttext_vectors)
)

word2vec_sim = cosine_similarity(
    [query_vector],
    np.vstack(word2vec_vectors)
)

FastText still produces embeddings for misspelled words because it uses subword units, while Word2Vec cannot represent words not in its vocabulary.

In [37]:
# vocabulary coverage
unique_words = set()

for tokens in processed_reviews:
    unique_words.update(tokens)

In [38]:
# count word3vec coverage
word2vec_count = 0

for word in unique_words:
    if word in word2vec:
        word2vec_count += 1

In [39]:
# fastext coverage
fasttext_count = 0

for word in unique_words:
    try:
        fasttext[word]
        fasttext_count += 1
    except:
        pass

In [40]:
# comparison
print("Total words:", len(unique_words))
print("Word2Vec coverage:", word2vec_count)
print("FastText coverage:", fasttext_count)

Total words: 114733
Word2Vec coverage: 47985
FastText coverage: 52613


fastext advantage: handles misspellings
rare words
new words

Word2Vec limitation
, If word not in vocabulary: vector does not exist